# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a Croissant-compliant dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset's metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Show metadata summary
print(f"Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {getattr(dataset.metadata, 'date_published', 'N/A')}")
print(f"Cite as: {getattr(dataset.metadata, 'cite_as', 'N/A')}")

## 2. Data Overview
Discover available record sets, fields, and their unique Croissant `@id`s.

In [ ]:
# List available record sets in the dataset (referenced by their '@id')
record_sets = dataset.metadata.record_sets
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set @id: {rs.id}")
    print(f"  Name: {getattr(rs, 'name', None)}")
    print(f"  Description: {getattr(rs, 'description', None)}")
    print(f"  Data sources: {[src.id for src in getattr(rs, 'sources', [])]}")
    print(f"  Fields:")
    for field in getattr(rs, 'fields', []):
        print(f"    - Field @id: {field.id}")
        print(f"      Name: {getattr(field, 'name', None)} | Data type: {getattr(field, 'data_type', None)}")
    print()

## 3. Data Extraction
Load records from a specific record set into a Pandas DataFrame using the record set and field `@id`s identified above.

In [ ]:
dataframes = {}

# Use the first available record set for illustration (replace with specific @id as needed)
if len(record_sets) > 0:
    record_set_id = record_sets[0].id
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Fields (columns) in record set '{record_set_id}':\n{df.columns.tolist()}")
    display(df.head())
else:
    print("No record sets found in dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering, normalizing, or aggregating over the records. Make sure to reference fields by their `@id` as given in the dataset.

In [ ]:
# Example EDA: Choose a numeric field by its @id (replace with field @id as needed)
if len(dataframes) > 0:
    # List fields and infer numeric candidates
    fields = {field.id: field for field in record_sets[0].fields}
    numeric_candidates = [fid for fid, field in fields.items() if field.data_type in ["Float", "Integer", "Number"]]
    print(f"Numeric fields (by @id): {numeric_candidates}")

    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Use first numeric field for demo
        df = dataframes[record_set_id]
        # Remove records with missing/non-numeric values in this field
        col = numeric_field
        df = df[pd.to_numeric(df[col], errors='coerce').notnull()].copy()
        df[col] = pd.to_numeric(df[col])
        threshold = df[col].mean()
        filtered_df = df[df[col] > threshold].copy()
        print(f"Filtered records in '{col}' (>{threshold:.2f}): {len(filtered_df)} found.")
        # Normalize
        filtered_df[f"{col}_normalized"] = (filtered_df[col] - filtered_df[col].mean()) / filtered_df[col].std()
        print(filtered_df[[col, f"{col}_normalized"]].head())

        # Try grouping by a likely categorical field (not the numeric)
        other_fields = [fid for fid in fields if fid != numeric_field]
        group_field = None
        for fid in other_fields:
            # Pick first non-numeric field
            if fields[fid].data_type not in ["Float", "Integer", "Number"]:
                group_field = fid
                break
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field)[col].mean()
            print(f"Grouped mean of {col} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field found for EDA in this record set.")
else:
    print("No data available to analyze.")

## 5. Visualization
Visualize the distribution of a numeric field or its relationship with another field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if len(dataframes) > 0 and numeric_candidates:
    df = dataframes[record_set_id]
    col = numeric_candidates[0]
    df = df[pd.to_numeric(df[col], errors='coerce').notnull()].copy()
    df[col] = pd.to_numeric(df[col])
    plt.figure(figsize=(8, 5))
    sns.histplot(df[col], bins=30, kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()
else:
    print("No plot generated (no numeric fields).")

## 6. Conclusion
This notebook demonstrated loading, exploring, and analyzing a Croissant-compatible dataset using only schema-defined `@id` references for each entity (record sets, fields, columns). The `mlcroissant` library enables structured and FAIR data access for downstream research.

- **Continue**: Adapt the record set and field `@id` references to your specific analysis task. For domain-informed transformations, cross-reference field definitions and data dictionaries from the Croissant schema.